# Hodi AI — YOLOv8 Training (Kaggle)
Make sure:
- Dataset `hodi-augmented` is added via Add Data
- Accelerator is set to **GPU T4 x2**
- Run all cells in order

In [6]:
# 1. Find images — auto-detect correct path
import os, glob, random
from pathlib import Path

WORK_DIR = '/kaggle/working'

# Use a recursive glob to find where the images actually live
# This searches for any folder under /kaggle/input that contains .jpg files
print("Searching for images in /kaggle/input...")
all_jpgs = glob.glob('/kaggle/input/**/*.jpg', recursive=True)

if not all_jpgs:
    # If no jpgs, check for pngs just in case
    all_jpgs = glob.glob('/kaggle/input/**/*.png', recursive=True)

if all_jpgs:
    # Get the directory of the first image found
    DATASET_DIR = os.path.dirname(all_jpgs[0])
    imgs = all_jpgs
    print(f'✅ Success! Found {len(imgs)} images.')
    print(f'📍 Dataset location: {DATASET_DIR}')
else:
    # Debug: Print the actual structure so you can see what's wrong
    print("❌ No images found. Current /kaggle/input structure:")
    for root, dirs, files in os.walk('/kaggle/input'):
        level = root.replace('/kaggle/input', '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        if level < 3:
            for f in files[:3]:
                print(f'{indent}  {f}')
    raise Exception('No images found — please check if the dataset is attached.')

Searching for images in /kaggle/input...
✅ Success! Found 38000 images.
📍 Dataset location: /kaggle/input/datasets/susanngatia/hodi-augmented


In [7]:
# 2. Create train/val split
random.seed(42)
random.shuffle(imgs)

n_val      = int(len(imgs) * 0.2)
val_imgs   = imgs[:n_val]
train_imgs = imgs[n_val:]

Path(f'{WORK_DIR}/train.txt').write_text('\n'.join(train_imgs))
Path(f'{WORK_DIR}/val.txt').write_text('\n'.join(val_imgs))
print(f'Train: {len(train_imgs)}  Val: {len(val_imgs)}')

Train: 30400  Val: 7600


In [8]:
# 3. Write data.yaml
yaml_path = f'{WORK_DIR}/data.yaml'
Path(yaml_path).write_text(f"""train: {WORK_DIR}/train.txt
val: {WORK_DIR}/val.txt

nc: 19
names:
  0: matatu
  1: nganya
  2: city_hoppa
  3: school_bus
  4: large_bus
  5: bodaboda
  6: bodaboda_no_helmet
  7: bodaboda_stage
  8: tuk_tuk
  9: mkokoteni
  10: traffic_marshal
  11: conductor
  12: matatu_stage
  13: passenger
  14: illegal_dumping
  15: garbage_pile
  16: traffic_lights
  17: road_sign
  18: pothole
""")
print('data.yaml written.')

data.yaml written.


In [9]:
# 4. Install ultralytics
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.4 MB/s eta 0:00:00a 0:00:01


In [10]:
# 5. Confirm GPU
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

CUDA available: True
Device: Tesla T4


In [11]:
# 6. Train
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project=f'{WORK_DIR}/runs',
    name='hodi_nairobi',
    exist_ok=True,
    pretrained=True,
    patience=15,
    save=True,
    plots=True,
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False,

In [17]:
# 7. Confirm best weights
best = f'{WORK_DIR}/runs/hodi_nairobi/weights/best.pt'
print('Best weights:', best)
print('Size:', os.path.getsize(best) // 1024, 'KB')

Best weights: /kaggle/working/runs/hodi_nairobi/weights/best.pt
Size: 6100 KB


In [6]:
import os

# Search for the best.pt file
for root, dirs, files in os.walk('/kaggle/working'):
    if 'best.pt' in files:
        print(f"Found it! Your file path is: {os.path.join(root, 'best.pt')}")

In [7]:
from IPython.display import FileLink

# Replace this with the path found in Step 1 if it differs
path_to_file = '/kaggle/working/runs/hodi_nairobi/weights/best.pt'

if os.path.exists(path_to_file):
    print("Click the link below to download your model to your computer:")
    display(FileLink(path_to_file))
else:
    print("File not found at that specific path. Please check Step 1.")

File not found at that specific path. Please check Step 1.


In [8]:
import os
from IPython.display import FileLink

# This will search the entire working directory for any file named best.pt
found_path = None
for root, dirs, files in os.walk('/kaggle/working'):
    if 'best.pt' in files:
        found_path = os.path.join(root, 'best.pt')
        break

if found_path:
    print(f"✅ Found it at: {found_path}")
    print("Click the link below to download:")
    display(FileLink(found_path))
else:
    # If it's not in /working, let's check if it's in a hidden directory
    print("❌ best.pt not found in /kaggle/working.")
    print("Listing top-level folders to help debug:")
    print(os.listdir('/kaggle/working'))

❌ best.pt not found in /kaggle/working.
Listing top-level folders to help debug:
['.virtual_documents']


In [9]:
import os

# This lists the output of previous successful versions if they are attached
for root, dirs, files in os.walk('/kaggle/input'):
    if 'best.pt' in files:
        print(f"✅ Found a model in an attached version: {os.path.join(root, 'best.pt')}")

# This checks your current interactive working directory (which we know is empty)
print(f"Current Working Directory contents: {os.listdir('/kaggle/working')}")

Current Working Directory contents: ['.virtual_documents']
